# Problem 4: 数据工具 (Data Utilities)

在本问题中，你将实现训练语言模型所需的数据处理函数。

## 4.1 批量采样 (Batching)

**目标**: 实现 `run_get_batch` 函数。

训练语言模型时，我们需要从数据集中随机采样小批量数据。

**语言模型训练设置**:

语言模型训练的目标是预测序列中的下一个 token。
给定一个 token 序列 $[x_1, x_2, x_3, ..., x_n]$，我们构造训练样本为：

- **输入 (inputs)**: $[x_1, x_2, x_3, ..., x_T]$
- **目标 (targets)**: $[x_2, x_3, x_4, ..., x_{T+1}]$

即，预测每个位置的下一个 token。

**批量采样步骤**:

1. **随机选择起始位置**:
   - 从数据集中随机选择 $batch\_size$ 个起始位置
   - 确保每个位置都有足够的后续 token（至少 $context\_length + 1$ 个）
   - 最大起始索引 = $len(dataset) - context\_length - 1$

2. **提取序列**:
   - 对于每个起始位置 $i$，提取序列 $dataset[i : i + context\_length + 1]$
   - 前面 $context\_length$ 个 token 作为输入
   - 后面 $context\_length$ 个 token 作为目标（偏移 1）

3. **转换为张量**:
   - 将 numpy 数组转换为 PyTorch 张量
   - 移动到指定设备（CPU 或 CUDA）

**示例**:

假设数据集为 `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`，`batch_size=2`，`context_length=3`：

随机选择起始位置：`[1, 5]`

从位置 1 提取：`[1, 2, 3, 4]` (4 = 3 + 1)
- inputs: `[1, 2, 3]`
- targets: `[2, 3, 4]`

从位置 5 提取：`[5, 6, 7, 8]`
- inputs: `[5, 6, 7]`
- targets: `[6, 7, 8]`

最终返回：
- inputs: `[[1, 2, 3], [5, 6, 7]]` (shape: [2, 3])
- targets: `[[2, 3, 4], [6, 7, 8]]` (shape: [2, 3])

**实现要求**:
- 输入数据集是一维 numpy 数组（token IDs）
- 返回两个 PyTorch 张量（inputs 和 targets）
- 张量形状为 `[batch_size, context_length]`
- 张量类型为 `torch.LongTensor`（整数）
- 随机采样使用均匀分布（每个位置等概率）

**对应函数**: `tests/adapters.py` 中的 `run_get_batch(dataset, batch_size, context_length, device)`

**参数**:
- `dataset`: 一维 numpy 数组，包含 token IDs
- `batch_size`: 批量大小
- `context_length`: 每个样本的上下文长度
- `device`: PyTorch 设备字符串（如 'cpu' 或 'cuda:0'）

**返回**:
- 元组 `(inputs, targets)`，每个都是 `torch.LongTensor`，形状为 `[batch_size, context_length]`

**提示**: 使用 `torch.randint` 生成随机起始位置。